<a href="https://colab.research.google.com/github/GitGirlie27/urdu-ocr-codesaviours-si26-Amna/blob/main/SI26_Week5_Amna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Force-install compatible versions in one clean step
!pip install -q -U "transformers>=4.40.0" "accelerate>=0.26.0" datasets evaluate gradio opencv-python-headless pillow sentencepiece standard-imghdr jiwer

In [1]:
# ==========================================
# 1. INSTALL PACKAGES (Run once, then restart runtime if prompted)
# ==========================================
!pip install -q -U "transformers>=4.40.0" "accelerate>=0.26.0" datasets evaluate gradio opencv-python-headless pillow sentencepiece standard-imghdr jiwer

import os
import glob
import zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Safe Hugging Face Imports
import transformers
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer
)
import gradio as gr

print(f"Transformers version: {transformers.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Transformers version: 5.14.1
Using device: cuda


In [2]:
# ==========================================
# 2. EXTRACT DATASET
# ==========================================
zip_path = "/content/Urdu OCR Dataset.zip"
extract_path = "/content/Urdu_OCR_Dataset"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset extracted successfully!")
else:
    print(f"Note: '{zip_path}' not found. Ensure dataset zip is uploaded to root.")

Dataset extracted successfully!


In [3]:
# ==========================================
# 3. PREPROCESSING FUNCTION & EXECUTION
# ==========================================
def preprocess_image(image_path, save_path=None):
    img = cv2.imread(image_path)
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    gray = cv2.fastNlMeansDenoising(gray, h=15)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )
    kernel = np.ones((2, 2), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    if save_path is not None:
        cv2.imwrite(save_path, binary)

    return binary

input_folder = "/content/Urdu_OCR_Dataset/Urdu OCR Dataset"
output_folder = "/content/processed"
os.makedirs(output_folder, exist_ok=True)

if os.path.exists(input_folder):
    image_paths = sorted(
        glob.glob(os.path.join(input_folder, "*.png")),
        key=lambda x: os.path.basename(x)
    )
    print(f"Preprocessing {len(image_paths)} images...")
    for image_path in image_paths:
        filename = os.path.basename(image_path)
        save_path = os.path.join(output_folder, filename)
        preprocess_image(image_path, save_path)
    print("Finished preprocessing.")


Preprocessing 209 images...
Finished preprocessing.


In [4]:
# ==========================================
# 4. LOAD LABELS & PREPARE SPLIT
# ==========================================
csv_path = "/content/Labels csv.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    print("\nDataset Sample:")
    print(df.head())

    train_df, test_df = train_test_split(
        df, test_size=0.20, random_state=42, shuffle=True
    )
    print(f"Train split: {len(train_df)} | Test split: {len(test_df)}")
else:
    print(f"Warning: {csv_path} not found. Please verify CSV file path.")



Dataset Sample:
                   Image_no                                               Text
0  /content/processed/0.png  پشاور، بنوں (نما ئندہ جنگ، اے ایف پی) بنوں میں...
1  /content/processed/1.png        اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس
2  /content/processed/2.png  کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ پولیس تشدد سے ا...
3  /content/processed/3.png  منشور علی جاں بحق اور 2 خواتین سمیت 14 افراد ز...
4  /content/processed/4.png  علاقے میں کرفیو نافذ کر دیا گیا ہے۔ مکینوں نے ...
Train split: 166 | Test split: 42


In [5]:
# 5. ASSEMBLE URDU-COMPATIBLE PROCESSOR & MODEL

# TrOCR default tokenizer is English-only (RoBERTa).
# For Urdu, we use a Multilingual Tokenizer (xlm-roberta-base) with Vision Encoder (ViT).

image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Combine into a single processor wrapper
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Build Vision-Encoder-Decoder model structure
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-224-in21k",
    "xlm-roberta-base"
)

# Configure Special Generation Tokens
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model = model.to(device)
print("Model and Urdu-compatible Tokenizer initialized successfully!")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForCausalLM LOAD REPORT from: xlm-roberta-base
Key                                                                   | Status     | 
----------------------------------------------------------------------+------------+-
roberta.pooler.dense.weight                                           | UNEXPECTED | 
roberta.pooler.dense.bias                                             | UNEXPECTED | 
roberta.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.key.weight         | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.value.bias         | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.query.bias       

Model and Urdu-compatible Tokenizer initialized successfully!


In [6]:
# 6. DATASET CLASS & DATALOADERS

class UrduOCRDataset(Dataset):
    def __init__(self, dataframe, processor, image_folder="/content/processed", max_target_length=128):
        self.data = dataframe.reset_index(drop=True)
        self.processor = processor
        self.image_folder = image_folder
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = str(self.data.loc[idx, "Image_no"])
        if not filename.endswith(".png"):
            filename += ".png"

        image_path = os.path.join(self.image_folder, filename)
        text = str(self.data.loc[idx, "Text"])

        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_target_length,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        # Mask padding tokens for loss calculation
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

if os.path.exists(csv_path):
    train_dataset = UrduOCRDataset(train_df, processor)
    test_dataset = UrduOCRDataset(test_df, processor)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=4)

In [7]:
# 7. TRAINING LOOP

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 5

if os.path.exists(csv_path):
    print("\nStarting Training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_tokens = 0
        total_tokens = 0

        print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

        for batch_idx, batch in enumerate(train_loader):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            predictions = outputs.logits.argmax(dim=-1)
            mask = labels != -100
            correct_tokens += ((predictions == labels) & mask).sum().item()
            total_tokens += mask.sum().item()

            if batch_idx % 5 == 0:
                acc = (100 * correct_tokens / total_tokens) if total_tokens > 0 else 0
                print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f} | Acc: {acc:.2f}%")

        avg_loss = total_loss / len(train_loader)
        epoch_acc = (100 * correct_tokens / total_tokens) if total_tokens > 0 else 0
        print(f"Summary Epoch {epoch+1} -> Avg Loss: {avg_loss:.4f} | Accuracy: {epoch_acc:.2f}%")


Starting Training...

--- Epoch 1/5 ---
Batch 0/42 | Loss: 30.0043 | Acc: 0.00%
Batch 5/42 | Loss: 8.7664 | Acc: 0.46%
Batch 10/42 | Loss: 7.9465 | Acc: 0.80%
Batch 15/42 | Loss: 7.5946 | Acc: 1.20%
Batch 20/42 | Loss: 7.5508 | Acc: 1.29%
Batch 25/42 | Loss: 7.3434 | Acc: 1.42%
Batch 30/42 | Loss: 7.4124 | Acc: 1.68%
Batch 35/42 | Loss: 6.4152 | Acc: 1.77%
Batch 40/42 | Loss: 7.4808 | Acc: 1.95%
Summary Epoch 1 -> Avg Loss: 8.4511 | Accuracy: 1.94%

--- Epoch 2/5 ---
Batch 0/42 | Loss: 7.1648 | Acc: 2.69%
Batch 5/42 | Loss: 7.1307 | Acc: 2.15%
Batch 10/42 | Loss: 7.2155 | Acc: 2.61%
Batch 15/42 | Loss: 6.6210 | Acc: 2.75%
Batch 20/42 | Loss: 7.3647 | Acc: 2.97%
Batch 25/42 | Loss: 7.2106 | Acc: 2.79%
Batch 30/42 | Loss: 7.2157 | Acc: 2.71%
Batch 35/42 | Loss: 6.4273 | Acc: 2.72%
Batch 40/42 | Loss: 6.5767 | Acc: 2.87%
Summary Epoch 2 -> Avg Loss: 6.9492 | Accuracy: 2.89%

--- Epoch 3/5 ---
Batch 0/42 | Loss: 6.9613 | Acc: 1.91%
Batch 5/42 | Loss: 6.9100 | Acc: 2.22%
Batch 10/42 | Loss

In [8]:
# 8. SAVE MODEL & PROCESSOR

save_dir = "/content/urdu_trocr_model"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
print(f"Model successfully saved to {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to /content/urdu_trocr_model


In [10]:
!pip install pytesseract

In [ ]:
import pytesseract
# ==========================================
# DEPENDENCIES CHECK
# ==========================================
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install -q pytesseract pillow gradio opencv-python-headless torch

import os
import cv2
import numpy as np
import torch
import pytesseract
from PIL import Image
import gradio as gr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# PREPROCESSING FUNCTION
# ==========================================
def preprocess_for_ocr(pil_image):
    """Clean and binarize the image for optimal OCR character detection."""
    # Convert PIL Image to OpenCV format
    img = np.array(pil_image)
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img

    # Contrast adjustment & Denoising
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(gray, h=15)

    # Adaptive Thresholding
    binary = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )

    # Convert back to PIL
    return Image.fromarray(binary).convert("RGB")


# ==========================================
# EXTRACTION FUNCTION WITH HYBRID FALLBACK
# ==========================================
def extract_urdu_text(input_image):
    if input_image is None:
        return "Please upload an image."

    # Convert numpy array / PIL image safely
    if not isinstance(input_image, Image.Image):
        pil_img = Image.fromarray(input_image)
    else:
        pil_img = input_image

    # Preprocess image
    processed_img = preprocess_for_ocr(pil_img)

    # Primary OCR Engine: PyTesseract tuned for Urdu (Nastaliq/Urdu script)
    # --psm 6 assumes a single uniform block of text
    custom_config = r"--oem 3 --psm 6"
    extracted_text = pytesseract.image_to_string(
        processed_img, lang="urd", config=custom_config
    ).strip()

    # If PyTesseract finds text, return it immediately
    if extracted_text:
        return extracted_text

    # Fallback attempt on original image if preprocessed version missed text
    raw_text = pytesseract.image_to_string(
        pil_img, lang="urd", config=r"--oem 3 --psm 3"
    ).strip()

    if raw_text:
        return raw_text

    return "Could not extract Urdu text. Please ensure the image is clear and high resolution."


# ==========================================
# GRADIO INTERFACE
# ==========================================
interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(label="Upload Urdu Image"),
    outputs=gr.Textbox(label="Extracted Urdu Text", lines=4),
    title="Urdu OCR System",
    description="Upload an image containing Urdu text to extract text.",
)

interface.launch(share=True, debug=True)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 2s (516 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False